# Difference-in-Differences: Implementation
**ECON 4370 — Applied Data Tools for Economics**  
**Dr. Fidel González — Spring 2026**

This notebook picks up exactly where the lecture ended.  
We translate the John Snow / Card & Krueger story into working Python code:

- Build the 2×2 table by hand
- Visualize parallel trends and the DiD estimate
- Run DiD as a regression (the standard applied approach)
- Test the parallel trends assumption with a pre-trends plot
- Break the estimator intentionally — so you understand when it fails

> **No external downloads required.** All data are simulated with calibrated parameters so you know the true answer before running the code.


---
## 0) Learning Objectives

By the end of this notebook you can:

1. Construct the **2×2 DiD table** and compute the estimator by hand.
2. Produce a **DiD visualization** that shows parallel pre-trends and the treatment effect.
3. Implement DiD as an **OLS regression** with a treatment indicator and interaction term.
4. Run a **pre-trends test** (event study) to evaluate the parallel trends assumption.
5. Explain what goes wrong when the assumption is violated.

> **Mindset for this notebook:** Write your prior *before* running each analysis. What do you expect to see? Being wrong is fine — being surprised without reflecting is not.


---
## 1) Setup


In [ ]:
# 1) Setup — libraries and folder structure
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

# Optional: statsmodels for regression output (install if needed)
try:
    import statsmodels.formula.api as smf
    STATSMODELS = True
except ImportError:
    STATSMODELS = False
    print("statsmodels not found — regression section will use numpy instead.")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.3f}".format)

# Folder structure (mirrors course convention)
ROOT      = Path.cwd() / "lecture_did"
RAW_DIR   = ROOT / "data_raw"
CLEAN_DIR = ROOT / "data_clean"
EXPORT_DIR= ROOT / "exports"

for d in [RAW_DIR, CLEAN_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Consistent plot style
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})

NAVY  = "#1A2B4A"
TEAL  = "#2A9D8F"
GOLD  = "#E9A42E"
RED   = "#C0392B"
GRAY  = "#8A9AAA"

ROOT

---
## 2) Simulate the Data — Card & Krueger Style

We simulate a dataset inspired by **Card & Krueger (1994)**:  
New Jersey raised its minimum wage; Pennsylvania did not.  
Did fast-food employment fall in New Jersey as a result?

**Calibration (you know the truth):**
- We observe restaurants in **NJ (treated)** and **PA (control)** at two time points
- There is a common upward time trend of **+1.5 employees** (economy-wide, affects both states)
- NJ has a higher baseline than PA (**levels differ** — parallel trends allows this)
- The **true treatment effect** is **−1.2 employees** (a small negative effect)
- Noise is added so the data looks realistic

**Your job:** recover that −1.2 using DiD.


In [ ]:
# 2) Simulate restaurant-level panel data
np.random.seed(42)

# --- Parameters (calibrated ground truth) ---
N_NJ          = 80    # number of NJ restaurants
N_PA          = 80    # number of PA restaurants
BASELINE_NJ   = 22.0  # avg employees in NJ before treatment
BASELINE_PA   = 19.5  # avg employees in PA before treatment (levels differ — that is OK)
TIME_TREND    = 1.5   # common time trend (affects BOTH states equally)
TRUE_EFFECT   = -1.2  # causal effect of the NJ minimum wage hike
NOISE_SD      = 2.8   # restaurant-level noise

# --- Pre-period (period = 0, before the policy) ---
emp_nj_pre = BASELINE_NJ + np.random.normal(0, NOISE_SD, N_NJ)
emp_pa_pre = BASELINE_PA + np.random.normal(0, NOISE_SD, N_PA)

# --- Post-period (period = 1, after the policy) ---
# Both states move up by TIME_TREND; NJ also gets the treatment effect
emp_nj_post = BASELINE_NJ + TIME_TREND + TRUE_EFFECT + np.random.normal(0, NOISE_SD, N_NJ)
emp_pa_post = BASELINE_PA + TIME_TREND                + np.random.normal(0, NOISE_SD, N_PA)

# --- Assemble long-format panel (one row per restaurant-period) ---
df = pd.concat([
    pd.DataFrame({"restaurant_id": range(N_NJ),
                  "state": "NJ", "treated": 1,
                  "period": 0, "post": 0,
                  "employment": emp_nj_pre}),
    pd.DataFrame({"restaurant_id": range(N_PA),
                  "state": "PA", "treated": 0,
                  "period": 0, "post": 0,
                  "employment": emp_pa_pre}),
    pd.DataFrame({"restaurant_id": range(N_NJ),
                  "state": "NJ", "treated": 1,
                  "period": 1, "post": 1,
                  "employment": emp_nj_post}),
    pd.DataFrame({"restaurant_id": range(N_PA),
                  "state": "PA", "treated": 0,
                  "period": 1, "post": 1,
                  "employment": emp_pa_post}),
], ignore_index=True)

# Create the interaction term (this is the DiD coefficient in regression)
df["treated_x_post"] = df["treated"] * df["post"]

# Label for plots
df["period_label"] = df["period"].map({0: "Before", 1: "After"})

# Save as raw file
df.to_csv(RAW_DIR / "did_restaurants_raw.csv", index=False)

print(f"Shape: {df.shape}")
print(f"States: {df['state'].unique()}")
print(f"Periods: {df['period_label'].unique()}")
print(f"\nTrue DiD parameter = {TRUE_EFFECT}  (your DiD estimate should be close to this)")
df.groupby(["state", "period_label"])["employment"].mean().round(2).rename("mean_employment")

---
## 3) Explore the Raw Data

Before computing anything — get familiar with the structure.


In [ ]:
# 3.1) Quick structure check
print("=== Shape ===")
print(df.shape)

print("\n=== First rows ===")
display(df.head(8))

print("=== Counts by state × period ===")
display(df.groupby(["state", "period_label"]).size().rename("n_restaurants").reset_index())

In [ ]:
# 3.2) Summary statistics by group and period
summary = (df.groupby(["state", "period_label"])["employment"]
             .agg(["mean", "std", "min", "max", "count"])
             .round(3)
             .rename(columns={"mean": "Mean", "std": "SD",
                               "min": "Min", "max": "Max", "count": "N"}))
summary

---
## 4) Write Your Prior

**Before computing the DiD estimate, answer these questions:**

1. Just from the summary table above — which state has higher employment before the policy? After?
2. Do you expect NJ employment to go up or down in the post-period? By how much?
3. The true treatment effect is −1.2 employees. Is that economically large or small given baseline employment of ~22?
4. What do you expect the DiD regression coefficient to equal — positive or negative?

Write your answers here before moving on. ↓


**Your prior (write here):**

1. *TODO*
2. *TODO*
3. *TODO*
4. *TODO*


---
## 5) The 2×2 Table — DiD by Hand

The first step in any DiD analysis is always the 2×2 table.  
It is transparent, easy to communicate, and immediately tells you whether the DiD makes sense.

$$
\widehat{DiD} = \underbrace{(\bar{Y}_{NJ,post} - \bar{Y}_{NJ,pre})}_{\Delta_{NJ}} - \underbrace{(\bar{Y}_{PA,post} - \bar{Y}_{PA,pre})}_{\Delta_{PA}}
$$


In [ ]:
# 5.1) Compute the 2x2 table
means = (df.groupby(["state", "period_label"])["employment"]
           .mean()
           .unstack("period_label")   # columns = Before / After
           [["Before", "After"]])     # reorder columns

means["Difference"] = means["After"] - means["Before"]

# Add DiD row
did_row = means.loc["NJ"] - means.loc["PA"]
means.loc["DiD (NJ - PA)"] = did_row

print("=== 2×2 DiD Table (mean employment) ===")
display(means.round(3))

did_estimate = means.loc["DiD (NJ - PA)", "Difference"]
print(f"\nDiD estimate:  {did_estimate:.3f}")
print(f"True effect:   {TRUE_EFFECT}")
print(f"Difference:    {did_estimate - TRUE_EFFECT:.3f}  (sampling noise)")

**Interpretation cell — fill in:**

- $\Delta_{NJ}$ (NJ's within-state change) = `TODO`
- $\Delta_{PA}$ (PA's within-state change) = `TODO`
- DiD = `TODO`  
- The time trend (common shock) in PA's difference tells us that even without the minimum wage hike, employment was moving by `TODO` employees. The DiD strips that out.


---
## 6) Visualize DiD — Graph 1: The Standard DiD Plot

The plot shows:  
1. The actual paths for both groups (Before → After)  
2. The **counterfactual** for NJ: where NJ *would have gone* if it followed PA's trend  
3. The DiD estimate = gap between actual NJ post and counterfactual NJ post


In [ ]:
# 6) DiD visualization
# Get the 4 cell means
nj_pre  = df.query("state == 'NJ' and period == 0")["employment"].mean()
nj_post = df.query("state == 'NJ' and period == 1")["employment"].mean()
pa_pre  = df.query("state == 'PA' and period == 0")["employment"].mean()
pa_post = df.query("state == 'PA' and period == 1")["employment"].mean()

# Counterfactual: NJ follows PA's trend
nj_cf = nj_pre + (pa_post - pa_pre)

fig, ax = plt.subplots(figsize=(8, 5))

x = [0, 1]  # Before=0, After=1

# --- PA actual (control) ---
ax.plot(x, [pa_pre, pa_post], color=NAVY, lw=2.5, marker="o", ms=7, label="PA (Control) — actual")

# --- NJ actual (treated) ---
ax.plot(x, [nj_pre, nj_post], color=TEAL, lw=2.5, marker="o", ms=7, label="NJ (Treated) — actual")

# --- NJ counterfactual (dashed) ---
ax.plot(x, [nj_pre, nj_cf], color=TEAL, lw=1.8, ls="--", marker="o", ms=6,
        label="NJ (Treated) — counterfactual")

# --- DiD arrow ---
ax.annotate("", xy=(1, nj_post), xytext=(1, nj_cf),
            arrowprops=dict(arrowstyle="<->", color=GOLD, lw=2.5))
ax.text(1.04, (nj_post + nj_cf) / 2,
        f"DiD\n= {nj_post - nj_cf:.2f}",
        color=GOLD, fontsize=10, fontweight="bold", va="center")

# --- Parallel pre-trend annotation ---
ax.annotate("", xy=(0, nj_pre), xytext=(0, pa_pre),
            arrowprops=dict(arrowstyle="<->", color=GRAY, lw=1.2))
ax.text(-0.07, (nj_pre + pa_pre) / 2,
        "Level\ndifference\n(OK)",
        color=GRAY, fontsize=8, ha="right", va="center")

ax.set_xticks([0, 1])
ax.set_xticklabels(["Before\n(pre-policy)", "After\n(post-policy)"], fontsize=11)
ax.set_ylabel("Mean Employment (full-time equivalent)", fontsize=11)
ax.set_title("DiD: NJ vs PA Fast-Food Employment", fontsize=13, fontweight="bold", pad=14)
ax.set_xlim(-0.25, 1.45)
ax.legend(loc="lower right", fontsize=9)
ax.axvline(x=0.5, color=GRAY, lw=1, ls="dotted")
ax.text(0.51, ax.get_ylim()[0] + 0.2, "Policy →", color=GRAY, fontsize=8)

plt.tight_layout()
plt.savefig(EXPORT_DIR / "did_plot.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nNJ counterfactual post: {nj_cf:.3f}")
print(f"NJ actual post:        {nj_post:.3f}")
print(f"DiD (visual):          {nj_post - nj_cf:.3f}")

---
## 7) DiD as a Regression

In practice, economists run DiD as an OLS regression — not just the 2×2 table.  
This lets you add controls, cluster standard errors, and test hypotheses formally.

The regression model is:

$$
Y_{it} = \beta_0 + \beta_1 \cdot \text{Treated}_i + \beta_2 \cdot \text{Post}_t + \beta_3 \cdot (\text{Treated}_i \times \text{Post}_t) + \varepsilon_{it}
$$

| Coefficient | What it captures |
|-------------|------------------|
| $\beta_0$ | Baseline: PA in pre-period |
| $\beta_1$ | Level difference: NJ vs PA in pre-period |
| $\beta_2$ | Common time trend: PA post − PA pre |
| $\beta_3$ | **The DiD estimate: the treatment effect** |

> $\beta_3$ is the coefficient on the **interaction term** `treated × post`.  
> This is the number we care about.


In [ ]:
# 7.1) DiD regression using statsmodels
if STATSMODELS:
    model = smf.ols("employment ~ treated + post + treated_x_post", data=df)
    result = model.fit()
    print(result.summary())
else:
    # Fallback: numpy OLS (same point estimates, no standard errors)
    X = df[["treated", "post", "treated_x_post"]].copy()
    X.insert(0, "intercept", 1.0)
    y = df["employment"].values
    beta = np.linalg.lstsq(X.values, y, rcond=None)[0]
    coef_names = ["intercept", "treated", "post", "treated_x_post"]
    for name, b in zip(coef_names, beta):
        print(f"  {name:<20} {b:+.4f}")

In [ ]:
# 7.2) Extract and interpret the DiD coefficient
if STATSMODELS:
    beta_did = result.params["treated_x_post"]
    se_did   = result.bse["treated_x_post"]
    pval_did = result.pvalues["treated_x_post"]
    ci_lo, ci_hi = result.conf_int().loc["treated_x_post"]

    print("=== DiD Coefficient (β₃) ===")
    print(f"  Point estimate : {beta_did:.4f}")
    print(f"  Std. error     : {se_did:.4f}")
    print(f"  p-value        : {pval_did:.4f}")
    print(f"  95% CI         : [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"\n  True effect    : {TRUE_EFFECT}")
    print(f"  2x2 manual DiD : {did_estimate:.4f}")
    print("\n  → The OLS coefficient on treated_x_post equals the 2×2 table DiD exactly.")

**Interpretation cell — fill in:**

- $\hat{\beta}_0$ (baseline PA, pre-period) = `TODO` — does this match the raw mean you saw in Section 3?
- $\hat{\beta}_1$ (NJ − PA level gap) = `TODO` — is NJ higher or lower than PA at baseline?
- $\hat{\beta}_2$ (common time trend) = `TODO` — roughly equal to the calibrated `TIME_TREND = 1.5`?
- $\hat{\beta}_3$ (DiD estimate) = `TODO` — is the result statistically significant? Economically meaningful?
- Does the 95% CI contain the true parameter (−1.2)? `TODO`


---
## 8) Testing the Parallel Trends Assumption

We cannot directly test parallel trends for the post-period — that is fundamentally unobservable.  
But we **can** test whether the pre-treatment trends were parallel.  
If the two groups were diverging *before* the policy, the assumption is implausible.

The standard approach: extend the panel to include **multiple pre-periods** and run an event study.

We now simulate 3 pre-periods (t = −2, −1, 0) and 2 post-periods (t = 1, 2)  
and plot the treatment effect estimate at each period relative to the event.


In [ ]:
# 8.1) Simulate event-study panel (clean case — parallel pre-trends)
np.random.seed(99)

periods = [-2, -1, 0, 1, 2]  # 0 = year of treatment
event_rows = []

for t in periods:
    is_post = int(t >= 1)  # treatment kicks in at t=1

    for state, base, treated in [("NJ", BASELINE_NJ, 1), ("PA", BASELINE_PA, 0)]:
        n = N_NJ if state == "NJ" else N_PA
        # Common trend: 1.5 per period
        trend_effect = TIME_TREND * t
        # Treatment effect only applies to NJ in post periods
        treat_effect = TRUE_EFFECT * is_post * treated

        emp = base + trend_effect + treat_effect + np.random.normal(0, NOISE_SD, n)

        for e in emp:
            event_rows.append({
                "state": state, "treated": treated,
                "period": t, "post": is_post, "employment": e
            })

df_event = pd.DataFrame(event_rows)
df_event["treated_x_post"] = df_event["treated"] * df_event["post"]

# Group means by period and state
ev_means = df_event.groupby(["state", "period"])["employment"].mean().unstack("state")
ev_means["NJ - PA"] = ev_means["NJ"] - ev_means["PA"]

print("=== Group means by period ===")
display(ev_means.round(3))

In [ ]:
# 8.2) Event-study plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left panel: raw trajectories ---
ax = axes[0]
ax.plot(periods, ev_means["PA"], color=NAVY, lw=2.5, marker="o", ms=6, label="PA (Control)")
ax.plot(periods, ev_means["NJ"], color=TEAL, lw=2.5, marker="o", ms=6, label="NJ (Treated)")
ax.axvline(0.5, color=GRAY, ls="dotted", lw=1.2)
ax.text(0.55, ax.get_ylim()[0] + 0.5, "Policy →", color=GRAY, fontsize=9)
ax.set_xlabel("Period (0 = policy year)", fontsize=11)
ax.set_ylabel("Mean Employment", fontsize=11)
ax.set_title("Group Trajectories Over Time", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

# Annotate parallel pre-trends
ax.annotate("Parallel pre-trends",
            xy=(-1, ev_means.loc[-1, "NJ"]), xytext=(-2.0, ev_means.loc[-1, "NJ"] + 1.8),
            arrowprops=dict(arrowstyle="->", color=TEAL, lw=1.2),
            fontsize=8, color=TEAL)

# --- Right panel: NJ - PA gap by period (should be flat pre, then shift post) ---
ax2 = axes[1]
gap = ev_means["NJ - PA"]
pre_gap_mean = gap.loc[periods[0]:0].mean()

colors = [GRAY if t < 1 else TEAL for t in periods]
ax2.bar(periods, gap, color=colors, edgecolor="white", width=0.6, zorder=3)
ax2.axhline(pre_gap_mean, color=NAVY, ls="--", lw=1.5, label=f"Pre-period avg gap = {pre_gap_mean:.2f}")
ax2.axvline(0.5, color=GRAY, ls="dotted", lw=1.2)
ax2.axhline(0, color="black", lw=0.8)

ax2.set_xlabel("Period (0 = policy year)", fontsize=11)
ax2.set_ylabel("NJ − PA Gap", fontsize=11)
ax2.set_title("Pre-Trends Test: NJ vs PA Gap by Period", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)

# Annotate pre vs post
ax2.text(-1.5, gap.min() - 0.8, "Pre-period: flat gap\n→ parallel trends ✓",
         color=GRAY, fontsize=8, ha="center")
ax2.text(1.5, gap.max() + 0.5, "Post: shift downward\n→ negative effect",
         color=TEAL, fontsize=8, ha="center")

plt.suptitle("Event Study: Parallel Trends Test", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "event_study_clean.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n→ Pre-period gaps should be roughly constant.")
print("→ A jump after the treatment line = the DiD effect.")
print("→ If the pre-periods showed a growing gap — parallel trends would be implausible.")

**Interpretation cell — fill in:**

- Are the pre-period gaps (t = −2, −1, 0) roughly stable? `TODO`
- In the right panel, what happens to the gap at t = 1 and t = 2? `TODO`
- If a reviewer saw non-flat pre-period bars, what would they conclude about your DiD? `TODO`


---
## 9) Break the Estimator — Violate Parallel Trends

One of the most important things you can do as an analyst: understand **what failure looks like**.  
We now simulate a case where NJ was already trending downward *before* the policy.  
The DiD will give you the wrong answer — and we will see by exactly how much.


In [ ]:
# 9.1) Simulate the BROKEN case — NJ has a pre-existing downward drift
np.random.seed(77)

NJ_DRIFT = -0.8   # NJ was already declining by 0.8/period before the policy
                  # (the true policy effect is still TRUE_EFFECT = -1.2)

broken_rows = []
for t in periods:
    is_post = int(t >= 1)
    for state, base, treated in [("NJ", BASELINE_NJ, 1), ("PA", BASELINE_PA, 0)]:
        n = N_NJ if state == "NJ" else N_PA
        trend_effect   = TIME_TREND * t
        # NJ has an ADDITIONAL pre-existing drift (only applies before treatment)
        pre_drift      = NJ_DRIFT * t * treated * (1 - is_post)
        treat_effect   = TRUE_EFFECT * is_post * treated

        emp = base + trend_effect + pre_drift + treat_effect + np.random.normal(0, NOISE_SD, n)
        for e in emp:
            broken_rows.append({
                "state": state, "treated": treated,
                "period": t, "post": is_post, "employment": e
            })

df_broken = pd.DataFrame(broken_rows)
df_broken["treated_x_post"] = df_broken["treated"] * df_broken["post"]

# Compute the NAIVE DiD using only periods 0 and 1 (as if we had no pre-trend data)
naive_means = (df_broken.query("period in [0, 1]")
               .groupby(["state", "period"])["employment"].mean()
               .unstack("period"))
naive_means.columns = ["Before", "After"]
naive_means["Diff"] = naive_means["After"] - naive_means["Before"]

biased_did = naive_means.loc["NJ", "Diff"] - naive_means.loc["PA", "Diff"]

print("=== BIASED 2×2 Table ===")
display(naive_means.round(3))
print(f"\nBiased DiD estimate : {biased_did:.3f}")
print(f"True effect         : {TRUE_EFFECT}")
print(f"Bias                : {biased_did - TRUE_EFFECT:.3f}")
print(f"\n→ The DiD is OVERSTATING the negative effect.")
print(f"→ Why? Because NJ was already declining before the policy.")
print(f"   The estimator cannot distinguish the drift from the policy.")

In [ ]:
# 9.2) Plot: clean vs. broken pre-trends side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

def plot_trajectories(ax, df_plot, title, highlight_broken=False):
    gm = df_plot.groupby(["state", "period"])["employment"].mean().unstack("state")
    ax.plot(gm.index, gm["PA"], color=NAVY, lw=2.5, marker="o", ms=6, label="PA (Control)")
    ax.plot(gm.index, gm["NJ"], color=RED if highlight_broken else TEAL,
            lw=2.5, marker="o", ms=6, label="NJ (Treated)")
    ax.axvline(0.5, color=GRAY, ls="dotted", lw=1.2)
    if highlight_broken:
        ax.axvspan(-2.5, 0.5, alpha=0.05, color=RED)
        ax.text(-1.0, gm["NJ"].max() + 0.3,
                "Pre-treatment trend\nalready diverging!",
                color=RED, fontsize=8.5, ha="center")
    ax.set_xlabel("Period (0 = policy year)", fontsize=11)
    ax.set_ylabel("Mean Employment", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)

plot_trajectories(axes[0], df_event,  "Clean Case: Parallel Pre-Trends ✓",  highlight_broken=False)
plot_trajectories(axes[1], df_broken, "Broken Case: Non-Parallel Pre-Trends ✗", highlight_broken=True)

plt.suptitle("When Parallel Trends Holds vs. Fails", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "parallel_trends_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 10) Your Turn

Now it is your turn to experiment. Answer **at least two** of the three questions below.

---

**Exercise A — Change the true effect and re-run:**  
Edit the `TRUE_EFFECT` in Section 2 to `+2.0` (a positive effect of the wage hike).  
Re-run Sections 2 through 7.  
- Does the 2×2 table recover approximately +2.0?  
- Does the regression coefficient change sign?  
- What changes in the visualization?


In [ ]:
# Exercise A — Your code here


**Exercise B — Vary the noise and observe how the p-value changes:**  
Set `TRUE_EFFECT = -1.2` (as original) and try `NOISE_SD = 0.5` and then `NOISE_SD = 8.0`.  
- How does the regression p-value change?  
- Does the point estimate change? Does the standard error?  
- Write a one-sentence explanation of what this illustrates about statistical power.


In [ ]:
# Exercise B — Your code here


**Exercise C — Evaluate a real-world DiD claim:**  
You are reviewing a paper that claims a drug treatment reduced hospital readmissions by 15 percentage points.  
The treated group had worse pre-treatment health outcomes than the control group.  
The authors only show post-period data.  

List **two specific concerns** you would raise about the parallel trends assumption in this study.  
What additional data or tests would you request?


**Answer (Exercise C):**

1. *TODO*
2. *TODO*

**Additional data/tests I would request:**

*TODO*


---
## 11) Export and Wrap Up


In [ ]:
# 11) Export cleaned datasets and print file manifest
df.to_csv(CLEAN_DIR / "did_2period_clean.csv", index=False)
df_event.to_csv(CLEAN_DIR / "did_event_study_clean.csv", index=False)
df_broken.to_csv(CLEAN_DIR / "did_event_study_broken.csv", index=False)

print("=== Files written ===")
for f in sorted(list(CLEAN_DIR.glob("*.csv")) + list(EXPORT_DIR.glob("*"))):
    print(f"  {f}")

print("\n=== Final DiD summary ===")
print(f"  True treatment effect       : {TRUE_EFFECT}")
print(f"  2×2 DiD estimate            : {did_estimate:.4f}")
if STATSMODELS:
    print(f"  Regression coefficient β₃   : {beta_did:.4f}")
    print(f"  p-value                     : {pval_did:.4f}")
    print(f"  95% CI                      : [{ci_lo:.4f}, {ci_hi:.4f}]")

---
## 12) Key Takeaways

1. **DiD = subtract twice.** First difference removes baseline level differences (state fixed effects). Second difference removes common time shocks (time fixed effects). What remains — under parallel trends — is the causal effect.

2. **The 2×2 table and the regression give the same point estimate.** In the simple case (two groups, two periods, no covariates), OLS on `treated × post` recovers exactly the 2×2 DiD. The regression is more flexible — you can add covariates, cluster SEs, or extend to multiple periods.

3. **Parallel trends is the assumption that does the work.** We cannot test it in the post-period. But we can check pre-trends: if the two groups were already diverging before the policy, the assumption is implausible — and the estimator will be biased.

4. **Bias is directional.** When the treated group was already trending downward, DiD overstates the negative effect. You can see exactly how much in the 'broken' simulation above.

5. **Statistical significance ≠ causal validity.** A highly significant DiD coefficient means nothing if the parallel trends assumption fails. Rigor is about defending the design, not just the p-value.

---
**Next:** Extending DiD to multiple periods and multiple treatment groups — staggered adoption DiD and the recent econometrics literature.


---
## Submission Checklist

Before submitting this notebook:

- [ ] All cells run top-to-bottom without errors (`Kernel → Restart & Run All`)
- [ ] All `TODO` cells in Sections 4, 7, 8 contain written answers
- [ ] At least 2 of the 3 Your Turn exercises are completed (Section 10)
- [ ] Exported files are present in `lecture_did/exports/`
- [ ] No hard-coded API keys or absolute paths (use `Path.cwd()` / relative paths)
